# Hosting LLMs: Three Ways to Run a Model
### LLMs on Local & Cloud

**Audience:** Students who finished the Tools & Environment module (a working environment: VS Code, Python, GitHub, Colab + Drive, an API key in a hidden variable, **and a NAIRR / Jetstream2 GPU VM**).

**Goal:** run an LLM **three ways** and feel the trade-offs:
1. **Cloud API** — someone else's model and GPU (recall from the Tools & Environment module); pay per token, data leaves you.
2. **Rent a GPU for a session** — a **Colab T4**; host a capable open model yourself, free but time-limited.
3. **A GPU you keep** — your **NAIRR VM**; self-host a model on a machine you control across the whole term.

> 💡 **Key idea (new this module):** ways 2 and 3 are *the same skill* — you host the model yourself on a **remote GPU**. Colab is a GPU you *borrow for a session*; the NAIRR VM is a GPU you *keep for the term*. In fact **the GPU-hosting code in this notebook runs unchanged on a Colab T4 or on your NAIRR VM** (open it over VS Code Remote-SSH and pick the VM's Python kernel).

> 🎬 **New to LLM vocabulary?** Watch this fast primer on the basic terms first: https://www.youtube.com/watch?v=GtfqAr9CAgg

> ⚡ **Runtime:** for the GPU section on Colab, set **Runtime → Change runtime type → T4 GPU**. To run the very same cells on your **NAIRR VM**, open this notebook over Remote-SSH and select the VM's kernel — its A100 slice already has CUDA.

# 🔌 0 — Check Your Runtime

- Confirm whether you have a GPU and which one
- On Colab: Runtime → Change runtime type → **T4 GPU**
- `nvidia-smi` shows the GPU; `torch.cuda` confirms PyTorch sees it
- No GPU? The local (Ollama/CPU) section still works

> ### 🔍 Deeper Explanation
>
> Before anything else, know what hardware you are on. Run nvidia-smi to see the GPU and how much memory it has — a Colab T4 gives you about 16 GB of vRAM, which is the budget that decides which models will fit. Then we ask PyTorch whether it can see the GPU; if CUDA is available, our model will run on the GPU and be fast, and if not, it falls back to CPU and will be slow. If you forgot to switch the runtime to T4, this is where you'll notice — go change it now and re-run.

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# 🗺️ 1 — Three Ways to Run an LLM

- **Cloud API** — someone else's service; you send text, pay per token, data leaves you
- **Rented GPU (Colab T4)** — a remote GPU you control **for a session**; free, time-limited
- **A GPU you keep (NAIRR VM)** — a remote GPU you control **all term**; self-host, persistent, private
- Ways 2 & 3 are the same skill on different remote GPUs — the code here runs on either
- A tiny model on your laptop CPU is a fourth, minor option (fine for 1–2B models; slow)
- This notebook runs the API + a self-hosted GPU model; `llm_compare.ipynb` self-hosts on the VM with Ollama

> ### 🔍 Deeper Explanation
>
> There are three places an LLM can run, and the whole module is about feeling the trade-offs. First, a cloud API like OpenAI or Anthropic — the easiest and most capable, but your data leaves your machine and you pay per token. The other two are really one skill wearing two hats: you host the model yourself on a remote GPU. A Colab T4 is a GPU you borrow for a session — free and capable, but it resets when the session ends. Your NAIRR VM is a GPU you keep for the whole term — same idea, but persistent, private, and yours to configure. The important thing to internalize: the GPU-hosting code in this notebook is identical whether it runs on Colab or on the VM. Running a small model on your laptop's CPU is a fourth, minor option — fine for a 1-to-2 billion model, but slow — so we treat the two remote GPUs as the main event.

In [ ]:
rows = [
    ("",            "Cloud API",        "Colab T4 (rent)",  "NAIRR VM (keep)"),
    ("Cost",        "per token ($)",    "free / session",   "free (allocation)"),
    ("Privacy",     "data leaves you",  "session on VM",    "your machine, private"),
    ("Persistence", "n/a (stateless)",  "resets each time", "persists all term"),
    ("Capability",  "highest",          "good (7-8B, T4)",  "good (14B+, A100 slice)"),
    ("Setup",       "an API key",       "pick T4, pip",     "provision once (setup)"),
]
for r in rows:
    print(f"{r[0]:13}{r[1]:18}{r[2]:18}{r[3]}")

# ☁️ 2 — Way 1: Cloud API (Recall from the Tools & Environment module)

- You already set up an API key in a hidden variable
- This is the baseline: most capable, simplest code
- But: per-token cost, and your prompt leaves your machine
- We'll compare local & Colab against this baseline

> ### 🔍 Deeper Explanation
>
> Let's anchor on what you already know. In the Tools & Environment module you stored an API key in a hidden variable and made a call. That hosted model is our baseline — the most capable and the least setup, just a few lines of code. The catch is the two things we'll keep coming back to: you pay per token, and your prompt is sent to a third party. The cell below makes one quick call if a key is present and otherwise skips, so the notebook runs either way. Keep this answer in mind; we'll ask the local and Colab models the same question and compare.

In [ ]:
import os
# Optional: paste your key into Colab Secrets (key icon) as OPENAI_API_KEY, or skip this cell.
try:
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
except Exception:
    pass

PROMPT = "In two sentences, what is SOC alert triage in cybersecurity?"

if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    r = client.chat.completions.create(model="gpt-4o-mini",
        messages=[{"role": "user", "content": PROMPT}])
    print("[API gpt-4o-mini]\n", r.choices[0].message.content)
else:
    print("No API key set — skipping the API baseline (that's fine).")

# 💻 3 — Way 2: Rent a GPU for a Session (Colab T4)

- The first self-hosting path: a **remote GPU you control for a session**
- On Colab: **Runtime → Change runtime type → T4 GPU** (16 GB vRAM, free)
- You load and run the model yourself with Hugging Face `transformers` (next section)
- Free and capable, but the session (and anything you installed) **resets** when it ends
- The very same code runs on your NAIRR VM (Way 3) — a GPU that *doesn't* reset
- *Aside:* a 1–2B model on your laptop CPU via Ollama also works — free & offline, just slow

> ### 🔍 Deeper Explanation
>
> Our first self-hosting path is a GPU you rent for a session — the Colab T4. You switch the runtime to a T4, and from there you load and run the model yourself instead of calling someone's API. It's free and genuinely capable, and the catch is impermanence: when the session ends, the machine and anything you installed are wiped. That's the one difference from Way 3, your NAIRR VM, which is the identical workflow on a GPU that persists. As an aside, you can also run a small one-to-two-billion model on your own laptop's CPU with Ollama — free, private, and offline, just slow — so keep it in your back pocket for tiny tasks. The next section does the real work: loading a capable open model on the GPU with four-bit quantization.

In [ ]:
# ---- Optional aside: a tiny model on your OWN laptop's CPU (no GPU needed) ----
# 1) Install Ollama from https://ollama.com  (Windows / macOS / Linux)
# 2) Pull and chat with a SMALL model that runs on CPU:
#       ollama run llama3.2:1b
#    then type your question, e.g. "In two sentences, what is SOC alert triage?"
# 3) This is the "fourth, minor" option — fine for 1-2B models, slow on CPU.
#    The MAIN self-hosting happens on a remote GPU (Colab T4 below, or your NAIRR VM).
print("Laptop-CPU aside above. Main event: self-host on a GPU (next cells).")

**Way 3 preview — self-host with Ollama on your NAIRR VM.** The persistent version of self-hosting lives in the companion notebook **`llm_compare.ipynb`** and the **Cloud VM / Private LLM labs**: you install Ollama on your VM, pull `llama3.1:8b` and `qwen2.5:14b`, and reach them from Python or a browser (Open WebUI) — a model server that's still there tomorrow. The optional cell below runs Ollama *inside Colab* just to show the same terminal-style interaction live (this uses the Colab machine, not your laptop):

In [ ]:
# Optional demo: Ollama in Colab (terminal/"!" style interaction)
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])      # start the server in the background
time.sleep(5)
!ollama run llama3.2:1b "In two sentences, what is SOC alert triage?"

# 🚀 4 — Host a Capable Model on a GPU (Colab T4 *or* your NAIRR VM)

- Load a **capable** open model from Hugging Face: Qwen2.5-7B-Instruct
- Fits a 16 GB T4 using **4-bit quantization** (bitsandbytes); an A100 slice fits it easily
- `transformers` downloads the weights and runs them on the GPU
- **This exact code runs on Colab or on your NAIRR VM** (open over Remote-SSH, pick the VM kernel)
- Open license (Apache-2.0), no gated-access token needed

> ### 🔍 Deeper Explanation
>
> Here's the heart of the lecture: hosting a genuinely capable model on Colab's free T4. We use Qwen2.5-7B-Instruct — a strong, openly licensed model — and load it in 4-bit precision with bitsandbytes so its weights shrink to about five gigabytes and fit comfortably in the T4's sixteen. The transformers library downloads the model from the Hugging Face Hub the first time, which takes a few minutes, then places it on the GPU. We deliberately picked an ungated model so nobody gets stuck on access tokens. If the download or memory ever feels too heavy, swap the model id for the smaller Qwen2.5-3B-Instruct — one line — and everything else still works.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"      # capable + open; fits a T4 in 4-bit
# Lighter alternative if you hit memory/time limits: "Qwen/Qwen2.5-3B-Instruct"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
print("Loaded", MODEL_ID, "on", model.device)

Now a small `chat()` helper, then ask it the **same** question we asked the API:

In [ ]:
def chat(prompt, system="You are a concise cybersecurity assistant.", max_new_tokens=256):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": prompt}]
    ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)

print(chat("In two sentences, what is SOC alert triage in cybersecurity?"))

# ⌨️ 5 — Three Ways to Interact

- **Python code** — call `chat(...)` (what we just did)
- **`!` in a cell** — run shell tools, e.g. `!nvidia-smi`, `!ollama run ...`
- **Terminal** — serve the model as a local API and `curl` it from the shell
- Same hosted model, three front doors — pick what fits your workflow

> ### 🔍 Deeper Explanation
>
> You can drive a model three ways, and you've now seen all of them. Inside Python you call a function like our chat helper — best when the model is one step in a bigger program, which is where this course is heading with agents. The exclamation mark in a notebook cell runs shell commands, like nvidia-smi to watch GPU memory, or ollama run to chat from the shell. And a plain terminal — on your laptop or on a NAIRR virtual machine later — runs those same commands. The interaction style is just convenience; the underlying model and prompt are identical. Let's watch GPU memory with a shell command while the model is loaded.

In [ ]:
# "!" shell interaction: check how much GPU memory the loaded model is using
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

# Python interaction: a security-flavored prompt
print(chat("List three signs that a login attempt might be a brute-force attack."))

**Talk to your model from the terminal.** Wrap the model you just hosted in a tiny local API running in the background, then call it with `curl` — exactly how an agent or another program will reach it later. Run the next cell once to start the server:

In [ ]:
# Serve the hosted model as a local API (runs in the background)
import threading, time
from flask import Flask, request, jsonify

api = Flask(__name__)

@api.route("/chat", methods=["POST"])
def _chat_endpoint():
    data = request.get_json(force=True)
    reply = chat(data.get("prompt", ""), max_new_tokens=int(data.get("max_new_tokens", 200)))
    return jsonify({"reply": reply})

threading.Thread(
    target=lambda: api.run(host="127.0.0.1", port=8000, threaded=True),
    daemon=True,
).start()
time.sleep(2)
print("Your LLM API is live at http://127.0.0.1:8000/chat")

Now interact with it using a **terminal command** (`curl` via `!`). Edit the prompt and re-run:

In [ ]:
%%bash
# a real terminal command talking to your hosted model:
curl -s -X POST http://127.0.0.1:8000/chat -H "Content-Type: application/json" \
     -d '{"prompt": "In one sentence, what is a firewall?"}'
echo

> That `curl` call is a real terminal command hitting the model you hosted. The same request works from a true shell (e.g., a VM's terminal) or from any program — which is how the agents in later modules will call your LLM.

# 🤗 6 — Explore the Hugging Face Hub

- huggingface.co/models — filter by **Text Generation**, sort by trending
- Read the **model card**: size, license, and whether it's *gated*
- Match the size to your GPU: ~7-8B fits a T4 in 4-bit; 1-3B is easy
- Open licenses (Apache-2.0, MIT) avoid access-token hassles
- Good families to try: Qwen2.5, Gemma 2, Phi-3.5, Mistral

> ### 🔍 Deeper Explanation
>
> The Hugging Face Hub is the app store of open models, and learning to shop there is a real skill. Go to huggingface.co/models, filter by the Text Generation task, and sort by trending or downloads to see what the community actually uses. Open each candidate's model card and check three things: the parameter size, which tells you if it fits your GPU; the license, where Apache-2.0 or MIT mean you can use it freely; and whether it's gated, which would require requesting access. For a T4, a 7-to-8-billion model in 4-bit is the sweet spot, and a 1-to-3 billion model is effortless. The cell lists a few trending text-generation models to get you started, but browsing the site yourself is the point.

In [ ]:
# List a few popular text-generation models to browse (or just visit huggingface.co/models)
try:
    from huggingface_hub import HfApi
    for m in HfApi().list_models(filter="text-generation", sort="downloads",
                                 direction=-1, limit=12):
        print(m.id)
except Exception as e:
    print("Browse huggingface.co/models directly. (Hub listing skipped:", e, ")")

# 🔁 7 — Swap In Your Model & Compare

- Pick one model from the Hub and load it the **same way**
- Run the **same prompt** on both; compare quality, speed, size
- Time the generation — tokens per second tells the speed story
- Bigger ≠ always better for a given task
- This is the comparison your assignment asks for

> ### 🔍 Deeper Explanation
>
> Now make it yours. Choose a model you found on the Hub — here we use Phi-3.5-mini as a smaller, faster contrast — and load it with the exact same code, just a different model id. Then run the same prompt through both and compare on three axes: answer quality, speed, and memory footprint. We time the generation so you can talk about tokens per second, not just vibes. You'll often find a smaller model is plenty for a given task and noticeably faster — which is the whole argument for picking the right tool rather than the biggest one. Capture what you observe; it goes straight into your comparison report and the forum discussion.

In [ ]:
import time

def load_pipe(model_id):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16)
    t = AutoTokenizer.from_pretrained(model_id)
    m = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
    return t, m

def timed_chat(t, m, prompt, max_new_tokens=200):
    ids = t.apply_chat_template([{"role": "user", "content": prompt}],
                                add_generation_prompt=True, return_tensors="pt").to(m.device)
    start = time.time()
    out = m.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    secs = time.time() - start
    n = out.shape[-1] - ids.shape[-1]
    text = t.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
    print(f"  {n} tokens in {secs:.1f}s  ->  {n/secs:.1f} tok/s")
    return text

# Your pick from the Hub (swap this id for the model YOU chose):
MY_PICK = "microsoft/Phi-3.5-mini-instruct"
t2, m2 = load_pipe(MY_PICK)

q = "Explain what a CVE is to a new analyst, in three sentences."
print("=== Qwen2.5-7B ==="); print(timed_chat(tok, model, q))
print("\n=== " + MY_PICK + " ==="); print(timed_chat(t2, m2, q))

# ⚖️ 8 — Compare the Experiences

- **Cloud API** — most capable, simplest; per-token cost + data leaves you
- **Colab T4 (rent a GPU)** — capable, free session; setup + everything resets
- **NAIRR VM (a GPU you keep)** — capable, private, persistent all term; you manage it
- *Laptop CPU* — free & offline, but only tiny models
- Right tool depends on the task: privacy? cost? persistence? capability?
- → Write this up in your lab report and the forum

> ### 🔍 Deeper Explanation
>
> Step back and compare the three experiences. The cloud API was the most capable and the simplest code, but you pay per token and your data leaves your machine. The Colab T4 let you host a capable model for free — but it's a rental: the moment the session ends, it's gone. Your NAIRR VM is the same self-hosting workflow on a GPU you keep — private, persistent, and yours to configure — at the cost of managing it yourself. A tiny model on your laptop CPU rounds things out for offline odd jobs. There's no universal winner: a privacy-sensitive log argues for the VM, a quick hard question argues for the API, and a free throwaway experiment argues for Colab. That judgment is exactly what your comparison report and the forum post assess.

In [ ]:
print("Decision guide:")
print("  Max capability, quick one-off        -> cloud API (watch token cost)")
print("  Free capable experiment, throwaway    -> Colab T4 (rent a GPU)")
print("  Private, persistent, all-term server  -> your NAIRR VM (a GPU you keep)")
print("  Tiny task, fully offline              -> small model on your laptop CPU")
print("\nYou ran all three. Now write up which you'd pick for which task.")

- You ran an LLM **three ways**: a **cloud API**, a **rented GPU (Colab T4)**, and **a GPU you keep (NAIRR VM)**
- Ways 2 & 3 were the same self-hosting code on two different remote GPUs
- You hosted a **capable** open model on a GPU with 4-bit quantization
- You learned to **shop the Hugging Face Hub** (size, license, gated)
- Trade-offs — cost, privacy, persistence, capability — drive the choice

> ### 🔍 Deeper Explanation
>
> In forty minutes you've done what used to take a research lab: run a capable language model on a free GPU, run a private one on your own machine, and compare both to a hosted API. You also learned to navigate the Hugging Face Hub and read a model card for the three things that matter — size, license, and access. Carry forward the core lesson: there's no single best place to run an LLM; cost, privacy, and capability decide. In the next modules we'll take the model you can now host and put it to work — grounding it with RAG and driving it with agents.

In [ ]:
print("You've completed the LLMs on Local & Cloud module. You can now:")
print("  - run an LLM via API, locally on CPU, and on a Colab GPU")
print("  - host a capable open model on a T4 with 4-bit quantization")
print("  - choose a model from Hugging Face and compare it to others")
print("\nNext: ground these models with retrieval (RAG) and drive them with agents.")